# 00 — Setup & Config

Run this once per fresh Colab session. It:
1. Clones/pulls the GitHub repo into `/content` and adds `src/` to `sys.path`
2. Mounts Drive (data only — never committed to the repo)
3. Verifies the real folder/file structure on disk
4. Writes `configs/*.yaml` **only if they don't already exist** — if you've
   previously set and pushed real cutoff values, this will never overwrite
   them back to `null`.

Repo: https://github.com/AditPradana36/crash-dualgraph

In [ ]:
# ── Clone or update the repo (code lives in /content, not Drive) ──────────
REPO_URL = "https://github.com/AditPradana36/crash-dualgraph.git"
REPO_DIR = "/content/crash-dualgraph"

import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

import sys
sys.path.append(f"{REPO_DIR}/src")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install -q pyyaml tqdm pandas geopandas shapely

In [ ]:
import json
from pathlib import Path
from tqdm.auto import tqdm

BASE_DIR = Path("/content/drive/MyDrive/crash-dualgraph/data")

EXPECTED_PATHS = {
    "boundary_geojson":        BASE_DIR / "boundary" / "study-area.geojson",
    "positive_front_view_dir": BASE_DIR / "positive" / "front-view",
    "positive_metadata_csv":   BASE_DIR / "positive" / "panorama" / "metadata_outofrange_removed.csv",
    "positive_points_csv":     BASE_DIR / "positive" / "points" / "positive_points.csv",
    "negative_front_view_dir": BASE_DIR / "negative" / "front-view",
    "negative_metadata_csv":   BASE_DIR / "negative" / "panorama" / "metadata_outofrange_removed.csv",
    "negative_points_csv":     BASE_DIR / "negative" / "points" / "negative_points.csv",
}

In [ ]:
# ── Verify everything exists, report counts ─────────────────────────────
import pandas as pd

IMG_EXTS = {".jpg", ".jpeg", ".png"}
report = {}

for key, path in tqdm(EXPECTED_PATHS.items(), desc="Verifying paths"):
    exists = path.exists()
    entry = {"path": str(path), "exists": exists}

    if exists and path.is_dir():
        entry["n_images"] = sum(1 for p in path.iterdir() if p.suffix.lower() in IMG_EXTS)
    elif exists and path.suffix == ".csv":
        df = pd.read_csv(path)
        entry["n_rows"] = len(df)
        entry["columns"] = list(df.columns)
    elif exists and path.suffix == ".geojson":
        entry["size_bytes"] = path.stat().st_size

    report[key] = entry

print(json.dumps(report, indent=2, default=str))

missing = [k for k, v in report.items() if not v["exists"]]
if missing:
    raise FileNotFoundError(f"Missing paths, fix before continuing: {missing}")
print("\n✅ All expected paths found.")

In [ ]:
# ── Early cross-check (full reconciliation happens in 01) ───────────────
for cls in ["positive", "negative"]:
    pts = pd.read_csv(EXPECTED_PATHS[f"{cls}_points_csv"])
    meta = pd.read_csv(EXPECTED_PATHS[f"{cls}_metadata_csv"])
    n_images = report[f"{cls}_front_view_dir"]["n_images"]
    print(f"[{cls}]  points.csv: {len(pts)} | metadata.csv: {len(meta)} | images: {n_images}")
    if not (len(pts) == len(meta) == n_images):
        print(f"   ⚠️  Counts don't match — expected, this is exactly what 01 reconciles.")

In [ ]:
# ── Write configs — ONLY if they don't already exist in the cloned repo ──
# If you've already set real cutoff values and pushed them, this leaves
# those files completely untouched. It only creates what's missing.
import yaml

CONFIG_DIR = Path(REPO_DIR) / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

paths_cfg = {
    "base_dir": str(BASE_DIR),
    **{k: str(v) for k, v in EXPECTED_PATHS.items()},
    "interim_dir":   str(BASE_DIR / "interim"),
    "processed_dir": str(BASE_DIR / "processed"),
    "outputs_dir":   str(BASE_DIR / "outputs"),
    "random_seed": 42,
}

default_configs = {
    # paths.yaml is gitignored — always regenerated, safe to overwrite every time
    "paths.yaml": (paths_cfg, True),

    "svg_schema.yaml": ({
        "mapillary_checkpoint": "facebook/mask2former-swin-large-mapillary-vistas-panoptic",
        "node_types": ["ego", "signage", "light_pole", "road_marking", "building", "vegetation"],
        "tier1_thing_classes": [
            "Traffic Light", "Traffic Sign (Front)", "Traffic Sign (Back)",
            "Traffic Sign Frame", "Street Light", "Pole", "Utility Pole",
        ],
        "stuff_classes_for_island_labeling": [
            "Building", "Vegetation", "Crosswalk - Plain", "Lane Marking - General",
        ],
        "enclosure_crop_fraction_from_top": 0.40,
        "enclosure_classes": ["building", "wall", "fence", "vegetation"],
        "near_cutoff_d": 0.15,
        "mask_confidence_threshold": 0.5,
        "mask_min_area_fraction": 0.1,
    }, False),

    "tvg_schema.yaml": ({
        "isovist_radius_m": 50,
        "crash_history_threshold_m": 100,
        "n_rays": 360,
        "building_shape_metrics": [
            "area", "perimeter", "circular_compactness", "elongation",
            "orientation", "shape_index",
        ],
        "isovist_shape_metrics": ["area", "compactness", "occlusivity"],
        "peer_incident_attributes": ["severity", "recency"],
        "bbox_padding_m": 100,
    }, False),

    "model.yaml": ({
        "hidden_dim": 64,
        "heads": 4,
        "svg_layers": 2,
        "tvg_layers": 3,
        "dropout": 0.35,
        "scenarios": ["A", "B", "C", "D", "E", "F", "G"],
        "ablation_scope": ["B", "C", "D", "E", "F"],
    }, False),

    "eval.yaml": ({
        "k_folds": 5,
        "repeats": 3,
        "primary_metric": "pr_auc",
        "significance_test": "nadeau_bengio_or_wilcoxon",
        "correction": "holm_bonferroni_or_fdr",
        "pairs": [
            ["A", "B"], ["C", "A"], ["C", "B"],
            ["C", "D"], ["C", "E"], ["C", "F"], ["D", "E"], ["D", "F"], ["E", "F"],
            ["G", "A"], ["G", "B"], ["G", "C"], ["G", "D"], ["G", "E"], ["G", "F"],
            ["B", "B+"], ["C", "C+"], ["D", "D+"], ["E", "E+"], ["F", "F+"],
        ],
        "batch_size": 32,
        "epoch_cap": 200,
        "patience": 20,
    }, False),
}

written, skipped = [], []
for fname, (cfg, always_overwrite) in default_configs.items():
    fpath = CONFIG_DIR / fname
    if fpath.exists() and not always_overwrite:
        skipped.append(fname)
        continue
    with open(fpath, "w") as f:
        yaml.safe_dump(cfg, f, sort_keys=False)
    written.append(fname)

print(f"Written (new or regenerated): {written}")
print(f"Skipped (already existed, left untouched): {skipped}")

In [ ]:
# ── Final summary ────────────────────────────────────────────────────────
print("Environment check complete.")
print(f"Repo:  {REPO_DIR}")
print(f"Data:  {BASE_DIR}")
print()
print("Open items still requiring YOUR input (check configs/*.yaml for null):")
for fname in ["svg_schema.yaml", "tvg_schema.yaml", "model.yaml", "eval.yaml"]:
    with open(CONFIG_DIR / fname) as f:
        cfg = yaml.safe_load(f)
    for k, v in cfg.items():
        if v is None:
            print(f"   - {fname}: {k}")
print()
print("Reminder: if you edit any config value below, commit + push it from")
print("this repo so the decision is preserved for next session, e.g.:")
print(f"  !cd {REPO_DIR} && git add configs/ && git commit -m 'set cutoffs' && git push")
print()
print("Next: 01_data_prep_sampling.ipynb")